# Serving a registered model over REST

This is the fourth **advanced** tutorial, and the payoff of the previous one.

`g_model_registry.ipynb` ended on a promise: the `@champion` alias is not just documentation, it's an **address**, and the next notebook serves it. This is that notebook.

It assumes you've worked through `g_`, which registered `ca_housing_price` and put a `@champion` alias on the better version. We now stand a REST endpoint in front of that alias and call it like any other web API.

What this covers:

1. **Why REST serving** — the gap between loading a model in your own Python and exposing it to everyone else.
2. **`mlflow models serve`** — one command turns a registered model into a web server.
3. **The `/invocations` contract** — the JSON in, the JSON out, and how the model's *signature* is the schema both sides agree on.
4. **The Docker path** — packaging that same server as a container image for real deployment.

## The problem: `load_model` only works for people who can run your Python

Every model load so far looked like this:

```python
model = mlflow.pyfunc.load_model("models:/ca_housing_price@champion")
preds = model.predict(X_test)
```

That's fine when *you* — or a batch job you write — does the predicting. It quietly assumes the caller can `import mlflow`, unpickle a scikit-learn object, and has the right Python and library versions installed.

Production consumers rarely meet that assumption:

- The thing that needs a prediction is a **web frontend**, a **Java backend**, or a **dashboard** — code that can't (and shouldn't) import your training stack.
- Even another Python service shouldn't have to pin *your* exact `scikit-learn` version just to call your model.
- You want one **stable network endpoint** that stays put while the model behind it is retrained and re-promoted.

The lingua franca for service-to-service calls is **HTTP + JSON**. *Serving* wraps the model in a small web server: any client that can POST JSON gets predictions, and the model's Python — with the exact dependencies it was logged with — stays on the server side.

The concerns this serves: **deployment**, **language-agnostic integration**, and **dependency isolation**.

## What `mlflow models serve` is

`mlflow models serve` loads any pyfunc model from a model URI and launches a local web server in front of it. In MLflow 3 that server is **FastAPI running under uvicorn**; it exposes a tiny, fixed API:

| Endpoint | Method | Purpose |
|---|---|---|
| `/invocations` | `POST` | score a batch of rows — JSON in, predictions out |
| `/health` (alias `/ping`) | `GET` | liveness check — returns `200` when the model is loaded |
| `/version` | `GET` | the MLflow + model server version |

The crucial part for us: the model URI you serve can be **`models:/ca_housing_price@champion`**. You serve the *alias*, not a version number or a run hash — so the endpoint always fronts whatever `g_` last promoted.

## The serving command

**Missing from upstream tutorial:** the upstream quickstarts show `mlflow models serve` but gloss over the two things that actually trip up a first run — the tracking URI and the environment manager.

Run this **in a separate terminal** (it blocks, like the tracking server):

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5001
mlflow models serve -m "models:/ca_housing_price@champion" -p 5002 --env-manager local
```

Two flags earn their keep:

- **`MLFLOW_TRACKING_URI`** — a `models:/…@champion` URI is resolved by *asking the registry* which version the alias points at. The serve process is separate from your notebook and has no idea where your registry is unless you tell it. Without this, it falls back to a local `./mlruns` store, can't find the model, and exits.
- **`--env-manager local`** — by *default*, serve rebuilds the model's logged dependencies in a fresh `virtualenv` (also available: `uv`, `conda`). That's the right behavior in production: the model runs against the *exact* libraries it was trained with, isolated from whatever else is on the box. For this tutorial it just means a slow first start, so we serve in the current environment with `local`. **Remember the trade-off:** `local` is convenient but gives up the reproducibility guarantee — ship with `virtualenv`/`uv`/`conda` in production.

Leave it running. The cells below talk to it on port `5002`.

## Prerequisites

1. **Tracking server on `5001`** — same as the other advanced notebooks:
   ```bash
   mlflow ui --host 127.0.0.1 --port 5001
   ```
2. **A `ca_housing_price@champion` model** — produced by running `g_model_registry.ipynb`. The next cell checks for it and, if it's missing (you skipped `g_`), registers a small stand-in so this notebook runs on its own.
3. **The serving process on `5002`** — the command from the section above, in its own terminal.

No new dependencies: `requests` ships as an MLflow dependency, and `scikit-learn` is already installed.

In [1]:
import json

import pandas as pd
import requests
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature

TRACKING_URI = "http://127.0.0.1:5001"
SERVE_URL = "http://127.0.0.1:5002"
MODEL_NAME = "ca_housing_price"

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

# The same split as g_, so the rows we score below are genuinely held out.
raw = fetch_california_housing(as_frame=True)
X, y = raw.data, raw.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

## Step 1: Confirm the serving target exists

`mlflow models serve` was pointed at `models:/ca_housing_price@champion`. If you ran `g_`, that alias already resolves. If you skipped straight here, the cell below registers a small stand-in model and aliases it `@champion`, so the rest of the notebook has something to serve.

Either way: **the serve process must have been started *after* this model existed.** It resolves the alias once, at startup. If you started serving before registering, restart it.

In [2]:
from mlflow.exceptions import RestException

try:
    mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
    print(f"Found {MODEL_NAME}@champion -> version {mv.version} (from g_). Ready to serve.")
except RestException:
    print("No @champion alias found — registering a stand-in so this notebook is self-contained.")
    standin = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=0)
    standin.fit(X_train, y_train)
    sig = infer_signature(X_train, standin.predict(X_train))
    with mlflow.start_run(run_name="serving_standin"):
        info = mlflow.sklearn.log_model(
            standin, name="model", signature=sig, input_example=X_train.head(),
        )
    version = mlflow.register_model(info.model_uri, MODEL_NAME).version
    client.set_registered_model_alias(MODEL_NAME, "champion", version)
    print(f"Registered {MODEL_NAME} v{version} and aliased it @champion. "
          "If serve was already running, restart it so it picks this up.")

Found ca_housing_price@champion -> version 1 (from g_). Ready to serve.


## Step 2: The `/invocations` contract

The scoring endpoint speaks JSON. It accepts a batch of rows under one of four structural keys; the two you'll use for tabular data are:

| Key | Shape | When |
|---|---|---|
| `dataframe_split` | `{"columns": [...], "data": [[...], ...]}` | columns + rows, compact for wide tables |
| `dataframe_records` | `[{"col": val, ...}, ...]` | one JSON object per row, more readable |
| `inputs` / `instances` | raw arrays/tensors | non-tabular models (images, text) |

The response is always `{"predictions": [...]}`.

The **schema** of `columns`/`data` is not arbitrary — it's the model **signature** `g_` attached with `infer_signature(...)`. The signature is the contract: the server validates incoming JSON against it and rejects anything that doesn't match (Step 4 shows what that looks like). This is exactly why logging a signature back in `b_`/`f_` mattered — it's what makes the served endpoint self-describing and safe.

We score the first five held-out rows using the explicit `dataframe_split` form so you can see the columns and data the server receives.

In [3]:
sample = X_test.head()

payload = {
    "dataframe_split": {
        "columns": list(sample.columns),
        "data": sample.values.tolist(),
    }
}

try:
    resp = requests.post(f"{SERVE_URL}/invocations", json=payload, timeout=10)
    resp.raise_for_status()
    predictions = resp.json()["predictions"]
    for row, pred in zip(sample.index, predictions):
        print(f"row {row:>5}:  predicted MedHouseVal = {pred:.3f}")
except requests.exceptions.ConnectionError:
    print(f"Could not reach {SERVE_URL}. Is the serving process running on 5002? "
          "See the 'serving command' section above.")

row 14740:  predicted MedHouseVal = 1.436
row 10101:  predicted MedHouseVal = 2.589
row 20566:  predicted MedHouseVal = 1.437
row  2670:  predicted MedHouseVal = 0.847
row 15709:  predicted MedHouseVal = 4.408


## Step 3: The REST endpoint and in-process loading return the same model

A served endpoint isn't a different, approximate model — it's *the same artifact* behind an HTTP boundary. To prove it, load `@champion` in-process (the old `load_model` way) and compare its predictions to the REST ones on the same rows. They should match to floating-point precision.

In [4]:
local_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
local_preds = local_model.predict(sample)

comparison = pd.DataFrame({
    "in_process (load_model)": local_preds,
    "over_REST (/invocations)": predictions,
})
print(comparison.round(6).to_string(index=False))
print("\nidentical:", bool((comparison.iloc[:, 0].round(6) == comparison.iloc[:, 1].round(6)).all()))

 in_process (load_model)  over_REST (/invocations)
                1.436230                  1.436230
                2.589157                  2.589157
                1.437104                  1.437104
                0.847418                  0.847418
                4.408091                  4.408091

identical: True


## Step 4: Health checks and the signature as a guardrail

Two things every operator of a served model relies on:

- **`/health`** — what a load balancer or Kubernetes liveness probe polls to decide whether the server is ready for traffic. It returns `200` only once the model is loaded.
- **Signature enforcement** — send the wrong columns and the server rejects the request with a `400` and a `SCHEMA_ENFORCEMENT_FAILED` error that names the missing input, *instead of* silently returning garbage. That guardrail exists only because the model was logged with a signature.

Below we hit `/health`, then deliberately send a payload missing the `MedInc` column to see the contract reject it.

In [5]:
health = requests.get(f"{SERVE_URL}/health", timeout=10)
print(f"GET /health -> {health.status_code} {health.text.strip()!r}")

# Drop a required column to violate the signature.
bad = sample.drop(columns=["MedInc"])
bad_payload = {"dataframe_split": {"columns": list(bad.columns), "data": bad.values.tolist()}}
bad_resp = requests.post(f"{SERVE_URL}/invocations", json=bad_payload, timeout=10)
err = bad_resp.json()
print(f"\nPOST /invocations (missing 'MedInc') -> {bad_resp.status_code} {err['error_class']}")
# The message embeds the full schema; the last clause is the punchline.
print("  ", err["message"].split("Error: ")[-1])

GET /health -> 200 ''

POST /invocations (missing 'MedInc') -> 400 SCHEMA_ENFORCEMENT_FAILED
   Model is missing inputs ['MedInc'].


## Step 5: From `serve` to a container — the deployment path

`mlflow models serve` is the development-and-test form. To actually ship, you usually want an immutable, self-contained **container image** you can hand to Kubernetes, Cloud Run, ECS, or a colleague. MLflow builds one for you:

```bash
mlflow models build-docker \
  -m "models:/ca_housing_price@champion" \
  -n ca-housing-server
```

That image bakes in the model, its logged dependencies, and the same FastAPI server — so the container exposes the identical `/invocations` and `/health` endpoints you just used. Run it with:

```bash
docker run -p 5002:8080 ca-housing-server
```

(We don't build the image in this notebook — it needs Docker installed and takes a few minutes — but the request code in Steps 2–4 works against the container unchanged. That's the point: the REST contract is stable across `serve`, a container, and a managed endpoint.)

**The bigger picture — managed deployment targets.** For production you rarely run `docker run` by hand. MLflow has deployment plugins that take the *same* model URI and push it to a managed endpoint: [Amazon SageMaker](https://mlflow.org/docs/latest/ml/deployment/), [Azure ML](https://mlflow.org/docs/latest/ml/deployment/), and Kubernetes (via KServe). The model artifact and the `/invocations` contract stay the same; only the target changes.

## Step 6: How a promotion reaches the endpoint

Here is the indirection from `g_` paying off end-to-end:

1. You serve `models:/ca_housing_price@champion`.
2. In `g_`, a challenger wins and you move the `@champion` alias to the new version — **one API call**, no change to the serving command or any client.
3. The next time the server *resolves the alias* — i.e. on restart or redeploy — it loads the new version. Clients keep POSTing to the same URL and transparently get the better model.

The one caveat to internalize: **`serve` resolves the alias at load time, not per request.** A running server keeps the version it started with; moving the alias doesn't hot-swap it. Picking up a promotion means restarting the process (or rolling the deployment). In a real system, the registry can fire a **webhook** on alias changes to trigger exactly that redeploy — the automated version of the manual restart.

This closes the loop the repo has been building: **track** an experiment (`b`–`e`), **evaluate and gate** it (`f`), **register and promote** it (`g`), and **serve** the promoted alias to the world (`h`).

## Next steps

That completes the traditional-ML MLOps spine: tracking → evaluation → registry → serving.

- **[MLflow Deployment docs](https://mlflow.org/docs/latest/ml/deployment/)** — SageMaker, Azure ML, and Kubernetes targets that consume the same model URI.
- **[Serving inputs & the scoring protocol](https://mlflow.org/docs/latest/ml/deployment/deploy-model-locally/)** — the full `/invocations` payload reference, including `inputs`/`instances` for non-tabular models.
- **Batch inference** — for scoring large tables offline rather than over HTTP, `mlflow.pyfunc.spark_udf(...)` turns the same registered model into a Spark UDF.
- **Beyond traditional ML — the GenAI track.** Everything so far has been classical regression/classification. MLflow 3's other half is for LLMs and agents: **tracing** (capturing an app's full execution), `mlflow.genai.evaluate()` with LLM-as-judge metrics, and a prompt registry. That's the natural Part 2 of this repo. Start at the [GenAI docs](https://mlflow.org/docs/latest/genai/).